In [5]:
import os
import sys
import time
import torch
import yaml
import pandas as pd

from calflops import calculate_flops

# =========================================================
# NAFNet PATH
# =========================================================

NAFNET_ROOT = r"C:\DNN\deepLearning\NAFNet"

sys.path.append(NAFNET_ROOT)

from basicsr.models import create_model

# =========================================================
# YAML PATH
# =========================================================
# 아무 yml 하나만 사용
# 구조 정보만 필요하기 때문
# =========================================================

YAML_PATH = (
    r"C:\DNN\deepLearning\sidl_options\test\mixed\easy"
    r"\Baseline_net_g_5000.yml"
)

# =========================================================
# EXPERIMENT ROOT
# =========================================================

EXP_ROOT = (
    r"C:\DNN\deepLearning\NAFNet\experiments"
)

# =========================================================
# LOAD YAML
# =========================================================

with open(YAML_PATH, 'r') as f:
    opt = yaml.safe_load(f)

opt['is_train'] = False
opt['dist'] = False

# =========================================================
# RESULT
# =========================================================

results = []

# =========================================================
# SEARCH EXPERIMENTS
# =========================================================

for exp_name in os.listdir(EXP_ROOT):

    exp_path = os.path.join(EXP_ROOT, exp_name)

    if not os.path.isdir(exp_path):
        continue

    # =====================================================
    # difficulty parsing
    # =====================================================

    exp_lower = exp_name.lower()

    if "easy" in exp_lower:
        difficulty = "easy"

    elif "medium" in exp_lower:
        difficulty = "medium"

    elif "hard" in exp_lower:
        difficulty = "hard"

    else:
        difficulty = "unknown"

    # =====================================================
    # model folder
    # =====================================================

    models_path = os.path.join(
        exp_path,
        "models"
    )

    if not os.path.isdir(models_path):
        continue

    print("\n" + "=" * 60)
    print("EXPERIMENT :", exp_name)
    print("DIFFICULTY :", difficulty)
    print("=" * 60)

    # =====================================================
    # SORT MODEL FILES
    # =====================================================

    model_list = sorted(
        os.listdir(models_path),
        key=lambda x: (
            999999
            if "latest" in x
            else int(
                x.replace("net_g_", "")
                 .replace(".pth", "")
            )
        )
    )

    # =====================================================
    # MODEL LOOP
    # =====================================================

    for model_file in model_list:

        if not model_file.endswith(".pth"):
            continue

        ckpt_path = os.path.join(
            models_path,
            model_file
        )

        print("\nMODEL :", model_file)

        # =================================================
        # epoch parse
        # =================================================

        epoch = "latest"

        if "latest" not in model_file:

            try:
                epoch = int(
                    model_file.replace("net_g_", "")
                              .replace(".pth", "")
                )

            except:
                epoch = -1

        # =================================================
        # checkpoint load
        # =================================================

        if 'path' not in opt:
            opt['path'] = {}

        opt['path']['pretrain_network_g'] = ckpt_path

        # =================================================
        # build model
        # =================================================

        model_wrapper = create_model(opt)

        net_g = model_wrapper.net_g.eval().cuda()

        # =================================================
        # dummy input
        # =================================================

        input_shape = (1, 3, 512, 512)

        dummy_input = torch.randn(
            input_shape,
            device='cuda'
        )

        # =================================================
        # FLOPS / PARAMS
        # =================================================

        with torch.no_grad():

            flops, gmacs, params = calculate_flops(
                model=net_g,
                input_shape=input_shape,
                output_as_string=True,
                output_precision=4
            )

        # =================================================
        # WARMUP
        # =================================================

        for _ in range(10):

            with torch.no_grad():

                _ = net_g(dummy_input)

        torch.cuda.synchronize()

        # =================================================
        # LATENCY
        # =================================================

        N_RUNS = 100

        start = time.perf_counter()

        with torch.no_grad():

            for _ in range(N_RUNS):

                _ = net_g(dummy_input)

        torch.cuda.synchronize()

        end = time.perf_counter()

        latency_ms = (
            (end - start) / N_RUNS * 1000
        )

        # =================================================
        # SAVE RESULT
        # =================================================

        row = {
            'difficulty': difficulty,
            'experiment': exp_name,
            'epoch': epoch,
            'model': model_file,
            'flops': flops,
            'gmacs': gmacs,
            'params': params,
            'latency_ms': round(latency_ms, 4)
        }

        results.append(row)

        print(row)

        # =================================================
        # CLEANUP
        # =================================================

        del net_g
        del model_wrapper

        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()

# =========================================================
# DATAFRAME
# =========================================================

df = pd.DataFrame(results)

df = df.sort_values(
    by=['difficulty', 'epoch']
)

# =========================================================
# SAVE CSV
# =========================================================

save_path = (
    r"C:\DNN\deepLearning\NAFNet"
    r"\Baseline_benchmark_result.csv"
)

df.to_csv(save_path, index=False)

# =========================================================
# RESULT
# =========================================================

print("\n" + "=" * 60)
print("CSV SAVED")
print(save_path)
print("=" * 60)

print(df)

2026-06-08 02:27:19,152 INFO: Loading NAFNetLocal model from C:\DNN\deepLearning\NAFNet\experiments\NAFNet-mixed-easy-w16\models\net_g_1000.pth.
2026-06-08 02:27:19,249 INFO: Model [ImageRestorationModel] is created.



EXPERIMENT : NAFNet-mixed-easy-w16
DIFFICULTY : easy

MODEL : net_g_1000.pth
 load net keys <built-in method keys of collections.OrderedDict object at 0x0000025314C19E40>

------------------------------------- Calculate Flops Results -------------------------------------
Notations:
number of parameters (Params), number of multiply-accumulate operations(MACs),
number of floating-point operations (FLOPs), floating-point operations per second (FLOPS),
fwd FLOPs (model forward propagation FLOPs), bwd FLOPs (model backward propagation FLOPs),
default model backpropagation takes 2.00 times as much computation as forward propagation.

Total Training Params:                                                  1.97 M  
fwd MACs:                                                               9.1703 GMACs
fwd FLOPs:                                                              18.5265 GFLOPS
fwd+bwd MACs:                                                           27.511 GMACs
fwd+bwd FLOPs:           

2026-06-08 02:27:21,431 INFO: Loading NAFNetLocal model from C:\DNN\deepLearning\NAFNet\experiments\NAFNet-mixed-easy-w16\models\net_g_2000.pth.


{'difficulty': 'easy', 'experiment': 'NAFNet-mixed-easy-w16', 'epoch': 1000, 'model': 'net_g_1000.pth', 'flops': '18.5265 GFLOPS', 'gmacs': '9.1703 GMACs', 'params': '1.9696 M', 'latency_ms': 17.8552}

MODEL : net_g_2000.pth
 load net keys <built-in method keys of collections.OrderedDict object at 0x000002533863ADC0>


2026-06-08 02:27:21,744 INFO: Model [ImageRestorationModel] is created.


...

{'difficulty': 'medium', 'experiment': 'NAFNet-water-medium-w16', 'epoch': 18000, 'model': 'net_g_18000.pth', 'flops': '18.5265 GFLOPS', 'gmacs': '9.1703 GMACs', 'params': '1.9696 M', 'latency_ms': 16.0547}

MODEL : net_g_19000.pth
 load net keys <built-in method keys of collections.OrderedDict object at 0x0000025319E074C0>

------------------------------------- Calculate Flops Results -------------------------------------
Notations:
number of parameters (Params), number of multiply-accumulate operations(MACs),
number of floating-point operations (FLOPs), floating-point operations per second (FLOPS),
fwd FLOPs (model forward propagation FLOPs), bwd FLOPs (model backward propagation FLOPs),
default model backpropagation takes 2.00 times as much computation as forward propagation.

Total Training Params:                                                  1.97 M  
fwd MACs:                                                               9.1703 GMACs
fwd FLOPs:                                 

2026-06-08 02:31:46,818 INFO: Loading NAFNetLocal model from C:\DNN\deepLearning\NAFNet\experiments\NAFNet-water-medium-w16\models\net_g_20000.pth.
2026-06-08 02:31:46,892 INFO: Model [ImageRestorationModel] is created.


{'difficulty': 'medium', 'experiment': 'NAFNet-water-medium-w16', 'epoch': 19000, 'model': 'net_g_19000.pth', 'flops': '18.5265 GFLOPS', 'gmacs': '9.1703 GMACs', 'params': '1.9696 M', 'latency_ms': 16.2146}

MODEL : net_g_20000.pth
 load net keys <built-in method keys of collections.OrderedDict object at 0x000002532D16FAC0>

------------------------------------- Calculate Flops Results -------------------------------------
Notations:
number of parameters (Params), number of multiply-accumulate operations(MACs),
number of floating-point operations (FLOPs), floating-point operations per second (FLOPS),
fwd FLOPs (model forward propagation FLOPs), bwd FLOPs (model backward propagation FLOPs),
default model backpropagation takes 2.00 times as much computation as forward propagation.

Total Training Params:                                                  1.97 M  
fwd MACs:                                                               9.1703 GMACs
fwd FLOPs:                                 

2026-06-08 02:31:48,943 INFO: Loading NAFNetLocal model from C:\DNN\deepLearning\NAFNet\experiments\NAFNet-water-medium-w16\models\net_g_latest.pth.
2026-06-08 02:31:49,038 INFO: Model [ImageRestorationModel] is created.


{'difficulty': 'medium', 'experiment': 'NAFNet-water-medium-w16', 'epoch': 20000, 'model': 'net_g_20000.pth', 'flops': '18.5265 GFLOPS', 'gmacs': '9.1703 GMACs', 'params': '1.9696 M', 'latency_ms': 17.5288}

MODEL : net_g_latest.pth
 load net keys <built-in method keys of collections.OrderedDict object at 0x0000025314E8ABC0>

------------------------------------- Calculate Flops Results -------------------------------------
Notations:
number of parameters (Params), number of multiply-accumulate operations(MACs),
number of floating-point operations (FLOPs), floating-point operations per second (FLOPS),
fwd FLOPs (model forward propagation FLOPs), bwd FLOPs (model backward propagation FLOPs),
default model backpropagation takes 2.00 times as much computation as forward propagation.

Total Training Params:                                                  1.97 M  
fwd MACs:                                                               9.1703 GMACs
fwd FLOPs:                                